# 🧪 Testing & Evaluasi Akurasi Dataset Primer (Data1.csv - Data11.csv)

Notebook ini mengeksekusi pipeline pengolahan sinyal dan ekstraksi fitur yang persis digunakan pada **`main_gui.py`** dan **`loading_page.py`** untuk seluruh 11 file pengukuran primer di folder `data_primer/`.

### Parameter Vital Sign yang Dievaluasi:
1. **Heart Rate (HR)** - Deteksi R-Peak Pan-Tompkins dari sinyal ECG.
2. **Respiratory Rate (RR)** - Pipeline Enhanced Multi-EDR Spectral Fusion.
3. **Oxygen Saturation (SpO2)** - Dual-wavelength PPG (Red & IR) Ratio-of-Ratios.
4. **Core Body Temperature (CBT)** - Model Neraca Termal Dua Kompartemen terkalibrasi.
5. **Systolic Blood Pressure (SBP)** - Model Deep Learning BPNet TFLite.
6. **Diastolic Blood Pressure (DBP)** - Model Deep Learning BPNet TFLite.

In [10]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Set path ke root project TriaGo
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from processing_data.processing_data import ECGProcessor, PPGProcessor
from model.bpnet_inference import BPNetTflitePredictor

DATA_PRIMER_DIR = PROJECT_ROOT / "data_primer"
print(f"Folder Dataset Primer: {DATA_PRIMER_DIR}")

Folder Dataset Primer: C:\Users\Adyty\Documents\Farid ITS\TriaGo\data_primer


## 🔄 Ekstraksi Fitur Berdasarkan Pipeline Main GUI & BPNet Model

In [11]:
ecg_processor = ECGProcessor(target_fs=125)
ppg_processor = PPGProcessor(target_fs=125)
try:
    bpnet_predictor = BPNetTflitePredictor()
except Exception as e:
    print(f"[WARN] BPNet Predictor error ({e}), menggunakan fallback.")
    bpnet_predictor = None

files = sorted(list(DATA_PRIMER_DIR.glob("Data*.csv")))
results_list = []

for f in files:
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    
    raw_time = df["Time (s)"].to_numpy(dtype=float)
    raw_red = df["PPG_Red"].to_numpy(dtype=float)
    raw_ir = df["PPG_IR"].to_numpy(dtype=float)
    raw_ecg = df["ECG"].to_numpy(dtype=float)
    t_amb = df["Temp_Ambient"].to_numpy(dtype=float)
    t_skin = df["Temp_Object"].to_numpy(dtype=float)
    
    # Ground Truths
    gt_hr = float(df["HR_Ground_Truth"].iloc[0])
    gt_rr = float(df["Respiratory_Rate_Ground_Truth"].iloc[0])
    gt_spo2 = float(df["SpO2_Ground_Truth"].iloc[0])
    gt_temp = float(df["Body_Temperature_Ground_Truth"].iloc[0])
    gt_sbp = float(df["SBP_Ground_Truth"].iloc[0])
    gt_dbp = float(df["DBP_Ground_Truth"].iloc[0])
    
    # 1. Pipeline ECG (Downsample 125 Hz, Notch, Detrend, Lowpass & Savgol Filtering, R-Peak, HR, & RR)
    ecg_125, time_125 = ecg_processor.downsample(raw_ecg, raw_time, 400)
    sig_notch = ecg_processor.notch(ecg_125, freq=50.0, fs=125)
    sig_detrend = ecg_processor.detrending(sig_notch, fs=125)
    sig_lpf = ecg_processor.lowpass(sig_detrend, lowcut=35.0, fs=125)
    ecg_smooth = ecg_processor.savgol(sig_lpf, window_size=11, poly_order=2)
    
    r_peaks, _ = ecg_processor.detect_r_peaks(ecg_125, fs=125)
    hr_est = ecg_processor.calculate_heart_rate(r_peaks, fs=125)
    rr_est, _, _ = ecg_processor.calculate_respiration_rate(ecg_smooth, r_peaks, fs=125)
    
    # 2. Pipeline PPG (SpO2 & PI)
    ppg_res = ppg_processor.process_ppg(raw_time, raw_red, raw_ir, fs_orig=400)
    spo2_est = ppg_res["spo2"]
    ir_clean = ppg_res["ir_clean"]
    
    # 3. Pipeline CBT (Core Body Temperature Model)
    WEIGHT_CORE = 0.64
    WEIGHT_SKIN = 0.36
    k_env = WEIGHT_SKIN / WEIGHT_CORE
    C_OFFSET = 4.692
    
    mean_skin = float(np.mean(t_skin))
    mean_amb = float(np.mean(t_amb))
    temp_est = mean_skin + k_env * (mean_skin - mean_amb) + C_OFFSET
    temp_est = float(np.clip(temp_est, 30.0, 43.0))
    
    # 4. Pipeline Deep Learning BPNet (Tekanan Darah SBP & DBP)
    sbp_est, dbp_est = 120.0, 80.0
    if bpnet_predictor is not None:
        try:
            bp_res = bpnet_predictor.predict_recording(ecg_125=ecg_smooth, ppg_125=ir_clean, fs=125.0)
            if bp_res.get("sqa_passed"):
                sbp_est = float(bp_res["sbp"])
                dbp_est = float(bp_res["dbp"])
            else:
                sbp_est = float(bp_res.get("sbp", 120.0))
                dbp_est = float(bp_res.get("dbp", 80.0))
        except Exception:
            pass
            
    results_list.append({
        "File": f.name,
        "GT_HR": gt_hr, "Est_HR": round(hr_est, 1), "Err_HR": abs(hr_est - gt_hr),
        "GT_RR": gt_rr, "Est_RR": round(rr_est, 1), "Err_RR": abs(rr_est - gt_rr),
        "GT_SpO2": gt_spo2, "Est_SpO2": round(spo2_est, 1), "Err_SpO2": abs(spo2_est - gt_spo2),
        "GT_Temp": gt_temp, "Est_Temp": round(temp_est, 2), "Err_Temp": abs(temp_est - gt_temp),
        "GT_SBP": gt_sbp, "Est_SBP": round(sbp_est, 1), "Err_SBP": abs(sbp_est - gt_sbp),
        "GT_DBP": gt_dbp, "Est_DBP": round(dbp_est, 1), "Err_DBP": abs(dbp_est - gt_dbp),
    })

df_results = pd.DataFrame(results_list)
df_results


        EVALUASI PER-SEGMEN SIGNAL QUALITY ASSESSMENT (SQA)
 [Segmen #01] ( 0.0s - 10.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 110.4 vs PPG: 60.0)
 [Segmen #02] ( 2.0s - 12.0s) -> [OK] LOLOS SQA (SBP: 145.4, DBP: 83.8)
 [Segmen #03] ( 4.0s - 14.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 117.7 vs PPG: 98.7)
 [Segmen #04] ( 6.0s - 16.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 118.7 vs PPG: 102.3)
 [Segmen #05] ( 8.0s - 18.0s) -> [OK] LOLOS SQA (SBP: 134.2, DBP: 77.3)
 [Segmen #06] (10.0s - 20.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 123.5 vs PPG: 100.9)
 [Segmen #07] (12.0s - 22.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 121.6 vs PPG: 100.3)
 [Segmen #08] (14.0s - 24.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 129.7 vs PPG: 104.8)
 [Segmen #09] (16.0s - 26.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 128.2 vs PPG: 107.2)
 [Segmen #10] (18.0s - 28.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 122.0 vs PPG: 106.7)
 [Segmen #11] (20.0s - 30.0s) -> [REJECT] DITOLAK: HR Mismatch (ECG: 136.1 vs

,File,GT_HR,Est_HR,Err_HR,GT_RR,Est_RR,Err_RR,GT_SpO2,Est_SpO2,Err_SpO2,GT_Temp,Est_Temp,Err_Temp,GT_SBP,Est_SBP,Err_SBP,GT_DBP,Est_DBP,Err_DBP
0,Data1.csv,81.0,77.7,3.28,21.0,21.8,0.78,99.0,99.1,0.09,36.4,37.80,1.401907,105.0,139.8,34.8,68.0,80.6,12.6
1,Data10.csv,67.0,77.8,10.79,23.0,26.4,3.41,97.0,97.8,0.77,36.4,35.61,0.786853,103.0,123.4,20.4,74.0,66.8,7.2
2,Data11.csv,73.0,79.0,5.95,26.0,27.1,1.08,99.0,100.0,1.00,36.3,35.60,0.700663,107.0,134.2,27.2,69.0,78.0,9.0
3,Data2.csv,80.0,82.3,2.35,20.0,19.1,0.88,99.0,97.7,1.32,36.4,37.28,0.881540,105.0,120.0,15.0,71.0,80.0,9.0
4,Data3.csv,72.0,78.7,6.65,21.0,19.1,1.95,98.0,97.7,0.31,36.5,36.86,0.355281,106.0,120.0,14.0,77.0,80.0,3.0
5,Data4.csv,74.0,79.6,5.57,19.0,19.7,0.72,99.0,97.7,1.29,36.4,36.63,0.232944,112.0,120.0,8.0,73.0,80.0,7.0
6,Data5.csv,81.0,73.1,7.91,20.0,20.4,0.45,99.0,97.7,1.28,36.4,37.20,0.801116,111.0,120.0,9.0,76.0,80.0,4.0
7,Data6.csv,73.0,73.5,0.52,23.0,13.4,9.62,98.0,97.6,0.36,36.4,35.81,0.589889,103.0,110.8,7.8,76.0,65.0,11.0
8,Data7.csv,73.0,75.0,1.99,22.0,17.6,4.37,98.0,97.8,0.20,36.4,36.03,0.366633,102.0,124.3,22.3,66.0,71.9,5.9
9,Data8.csv,80.0,75.3,4.71,21.0,26.4,5.39,99.0,97.7,1.27,36.4,36.11,0.286242,106.0,113.2,7.2,69.0,67.6,1.4


## 📊 Ringkasan Metrik Evaluasi (MAE & Akurasi Seluruh Parameter)

In [12]:
summary_metrics = []

metrics_config = [
    ("Heart Rate (HR)", "GT_HR", "Err_HR", "bpm"),
    ("Respiratory Rate (RR)", "GT_RR", "Err_RR", "bpm"),
    ("Oxygen Saturation (SpO2)", "GT_SpO2", "Err_SpO2", "%"),
    ("Core Body Temp (CBT)", "GT_Temp", "Err_Temp", "°C"),
    ("Systolic BP (SBP)", "GT_SBP", "Err_SBP", "mmHg"),
    ("Diastolic BP (DBP)", "GT_DBP", "Err_DBP", "mmHg"),
]

for name, gt_col, err_col, unit in metrics_config:
    mae = df_results[err_col].mean()
    rmse = np.sqrt((df_results[err_col] ** 2).mean())
    mape = 100.0 * (df_results[err_col] / df_results[gt_col]).mean()
    accuracy = 100.0 - mape
    
    summary_metrics.append({
        "Parameter": name,
        "Satuan": unit,
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "MAPE (%)": round(mape, 3),
        "Akurasi (%)": round(accuracy, 3)
    })

df_summary = pd.DataFrame(summary_metrics)
df_summary

,Parameter,Satuan,MAE,RMSE,MAPE (%),Akurasi (%)
0,Heart Rate (HR),bpm,5.132,5.859,7.013,92.987
1,Respiratory Rate (RR),bpm,2.875,3.911,12.927,87.073
2,Oxygen Saturation (SpO2),%,0.808,0.931,0.819,99.181
3,Core Body Temp (CBT),°C,0.667,0.746,1.833,98.167
4,Systolic BP (SBP),mmHg,15.136,17.949,14.372,85.628
5,Diastolic BP (DBP),mmHg,6.464,7.418,9.062,90.938
